# 01 · Portfolio Overview and Data Checks
Inspect the synthetic securitized portfolio and confirm the data is internally consistent before modeling.

In [1]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, '../scripts')
pd.set_option('display.width', 140)
portfolio = pd.read_csv('../data/sample_portfolio.csv', parse_dates=['maturity'])

In [2]:
portfolio.head(10)

,id,sector,tranche,rating,notional,coupon,maturity,cpr,cdr,severity,spread_bps
0,TRN1000,Agency MBS,Pass-through,AGY,74900000,0.0297,2046-10-01,0.0631,0.0000,0.00,71
1,TRN1001,ABS,Junior,B,117700000,0.0822,2028-10-01,0.1208,0.0826,0.50,849
2,TRN1002,ABS,Mezzanine,BBB,37200000,0.0640,2029-10-01,0.1428,0.0223,0.50,369
3,TRN1003,ABS,Senior,AA,100600000,0.0500,2031-10-01,0.1266,0.0082,0.50,217
4,TRN1004,CMBS,Senior,AA,95600000,0.0445,2034-10-01,0.0075,0.0051,0.40,168
5,TRN1005,CMBS,Junior,BB,18400000,0.0883,2032-10-01,0.0083,0.0321,0.40,545
6,TRN1006,RMBS,Senior,AAA,47200000,0.0597,2043-10-01,0.0633,0.0020,0.35,136
7,TRN1007,CLO,Senior,AAA,100100000,0.0306,2031-10-01,0.1081,0.0049,0.35,69
8,TRN1008,ABS,Mezzanine,A,43900000,0.0466,2030-10-01,0.0858,0.0130,0.50,294
9,TRN1009,Agency MBS,Pass-through,AGY,61200000,0.0334,2045-10-01,0.0915,0.0000,0.00,66


## Data quality checks
No missing values, no duplicate IDs, valid ranges for rates, and agency paper carrying no credit-loss assumption.

In [3]:
checks = {
    'missing values': int(portfolio.isna().sum().sum()),
    'duplicate ids': int(portfolio['id'].duplicated().sum()),
    'cpr outside 0-100%': int((~portfolio['cpr'].between(0, 1)).sum()),
    'cdr outside 0-100%': int((~portfolio['cdr'].between(0, 1)).sum()),
    'agency rows with credit losses': int(((portfolio['sector'] == 'Agency MBS') & (portfolio['cdr'] > 0)).sum()),
}
pd.Series(checks, name='failures')

missing values                    0
duplicate ids                     0
cpr outside 0-100%                0
cdr outside 0-100%                0
agency rows with credit losses    0
Name: failures, dtype: int64

## Composition

In [4]:
comp = portfolio.groupby('sector').agg(tranches=('id', 'count'), notional_mm=('notional', lambda s: s.sum() / 1e6),
                                       avg_spread_bps=('spread_bps', 'mean'), avg_cpr=('cpr', 'mean'), avg_cdr=('cdr', 'mean'))
comp.round(3)

,tranches,notional_mm,avg_spread_bps,avg_cpr,avg_cdr
sector,,,,,
ABS,8,472.9,326.500,0.117,0.019
Agency MBS,9,641.9,62.000,0.076,0.000
CLO,10,514.1,154.100,0.149,0.014
CMBS,6,357.2,265.667,0.010,0.012
RMBS,7,481.2,219.714,0.066,0.007


In [5]:
spread_by_rating = portfolio.pivot_table(values='spread_bps', index='rating', columns='sector', aggfunc='mean')
order = [r for r in ['AGY', 'AAA', 'AA', 'A', 'BBB', 'BB', 'B'] if r in spread_by_rating.index]
spread_by_rating.loc[order].round(0)

sector,ABS,Agency MBS,CLO,CMBS,RMBS
rating,,,,,
AGY,NaN,62.0,NaN,NaN,NaN
AAA,201.0,NaN,83.0,NaN,120.0
AA,233.0,NaN,112.0,182.0,177.0
A,294.0,NaN,NaN,228.0,212.0
BBB,369.0,NaN,260.0,NaN,303.0
BB,NaN,NaN,475.0,545.0,NaN
B,849.0,NaN,NaN,NaN,NaN


Spreads widen as ratings fall, which is the relationship a pricing model depends on.